# FFT Self-Gravity Solver — Validation Notebook

This notebook re-runs the validation tests of the FFT self-gravity solver in
`athenak-multigrid` and checks the results **independently in Julia** (its own binary-file
reader, its own Laplacian, its own analytic solutions). Companion document:
`validation/fft_selfgravity_validation.pdf`.

**Prerequisites**
- The code built with `-D Athena_ENABLE_FFT=ON` (see the PDF, Sec. 4).
- Julia packages: `CairoMakie` (and `IJulia` to run this notebook). If missing:
  `using Pkg; Pkg.add(["CairoMakie", "IJulia"])`

**Structure**
1. Helpers: running AthenaK, parsing diagnostics, reading `.bin` outputs
2. Test 1 — periodic box: discrete consistency (machine precision)
3. Test 2 — shear bookkeeping: exactness at shear-periodic instants
4. Test 3 — shearing-wave analytic solution: order of accuracy
5. Test 4 — open vertical BC: slab potential
6. Test 5 — swing amplification vs linear theory (gravity + dynamics)
7. Playground: your own initial conditions

**One caveat up front**: AthenaK `.bin` snapshots store data in *single precision*
(float32). Independent checks computed here from files therefore bottom out around
`1e-6`–`1e-4` (they still catch any real bug, which would be `O(1)`). The
double-precision residuals printed by the `fft_poisson` problem generator
(`# FFT-POISSON ...` lines, parsed below) are the authoritative numbers.

In [ ]:
# ---------------- configuration: adjust paths if your tree lives elsewhere ----------
const REPO   = expanduser("~/Library/CloudStorage/Dropbox/Research/code/athenak-multigrid")
const ATHENA = joinpath(REPO, "build", "src", "athena")
const INPUTS = joinpath(REPO, "inputs", "tests")
const RUNDIR = joinpath(REPO, "validation", "run")
mkpath(RUNDIR)
isfile(ATHENA) || @warn "athena executable not found — build first (PDF Sec. 4)"
using CairoMakie, Printf
CairoMakie.activate!(type="png")          # static inline figures
set_theme!(Theme(fontsize=13, Axis=(xgridcolor=(:gray, 0.25), ygridcolor=(:gray, 0.25))))
# validated categorical palette (fixed order: blue, orange, aqua)
const C1, C2, C3 = "#2a78d6", "#eb6834", "#1baf7a";


In [ ]:
# ---------------- run AthenaK and collect the pgen's diagnostic lines ----------------
"""
    run_athena(input; overrides=String[])

Run the athena binary on `input` (a file in `inputs/tests/`) inside RUNDIR with
command-line `overrides` like `"problem/time0=0.5"`. Returns `(diag, stdout)` where
`diag` are the `# FFT-POISSON ...` lines.
"""
function run_athena(input::AbstractString; overrides=String[])
    cmd = Cmd(String[ATHENA, "-i", joinpath(INPUTS, input), overrides...])
    out = read(pipeline(Cmd(cmd; dir=RUNDIR); stderr=devnull), String)
    diag = filter(l -> startswith(l, "# FFT-POISSON"), split(out, '\n'))
    isempty(diag) && occursin("FATAL", out) && error("athena failed:\n" *
        join(filter(l -> occursin("ERROR", l) || occursin("FATAL", l),
                    split(out, '\n')), '\n'))
    return String.(diag), out
end

"Parse `key= value` pairs from a diagnostic line into a Dict."
parse_diag(line) = Dict(m.captures[1] => parse(Float64, m.captures[2])
                        for m in eachmatch(r"(\w+)=\s*([0-9.eE+-]+)", line));


## AthenaK `.bin` reader (pure Julia)

Reads format version 1.1 (uniform grid), assembles all meshblocks into global 3D arrays
indexed `[i, j, k]` (x fastest, Julia-natural). Every test below reads the `hydro_w`
(primitives: `dens`, `velx`, ...) and `grav_phi` snapshots this way.

In [ ]:
function read_athenak_bin(filename::AbstractString)
    open(filename, "r") do io
        startswith(readline(io), "Athena binary output") || error("not an AthenaK bin file")
        pheader_count = parse(Int, split(readline(io), "=")[end])
        ph = Dict{String,String}()
        for _ in 1:pheader_count-1
            k, v = strip.(split(readline(io), "="))
            ph[k] = v
        end
        locsize = parse(Int, ph["size of location"])
        varsize = parse(Int, ph["size of variable"])
        nvars = parse(Int, split(readline(io), "=")[end])
        var_list = String.(split(readline(io))[2:end])
        header_size = parse(Int, split(readline(io), "=")[end])
        header = String[]
        for ln in split(String(read(io, header_size)), '\n')
            s = strip(first(split(ln, '#')))
            isempty(s) || push!(header, s)
        end
        function get(block, key)
            blk = "<none>"
            for ln in header
                if startswith(ln, "<"); blk = ln; continue; end
                kv = split(ln, "="); length(kv) == 2 || continue
                blk == "<$block>" && strip(kv[1]) == key && return strip(kv[2])
            end
            error("no parameter $block/$key")
        end
        nghost = parse(Int, get("mesh", "nghost"))
        Nx = [parse(Int, get("mesh", "nx$d")) for d in 1:3]
        mesh = Dict(k => parse(Float64, get("mesh", k)) for k in
                    ("x1min","x1max","x2min","x2max","x3min","x3max"))
        locT = locsize == 8 ? Float64 : Float32
        varT = varsize == 8 ? Float64 : Float32
        vars = Dict(v => Array{Float64,3}(undef, Nx[1], Nx[2], Nx[3]) for v in var_list)
        while !eof(io)
            idx = Int.(reinterpret(Int32, read(io, 24))) .- nghost
            n1 = idx[2]-idx[1]+1; n2 = idx[4]-idx[3]+1; n3 = idx[6]-idx[5]+1
            logical = Int.(reinterpret(Int32, read(io, 16)))
            _ = reinterpret(locT, read(io, 6*locsize))          # block geometry (unused)
            raw = reinterpret(varT, read(io, n1*n2*n3*nvars*varsize))
            data = reshape(Float64.(raw), (n1, n2, n3, nvars))  # file is i-fastest
            i0 = logical[1]*n1; j0 = logical[2]*n2; k0 = logical[3]*n3
            for (vi, v) in enumerate(var_list)
                vars[v][i0+1:i0+n1, j0+1:j0+n2, k0+1:k0+n3] = data[:, :, :, vi]
            end
        end
        coords = [range(mesh["x$(d)min"], mesh["x$(d)max"]; length=Nx[d]+1)[1:end-1] .+
                  0.5*(mesh["x$(d)max"]-mesh["x$(d)min"])/Nx[d] for d in 1:3]
        return (vars=vars, x=collect(coords[1]), y=collect(coords[2]),
                z=collect(coords[3]), Nx=Nx, mesh=mesh, var_names=var_list)
    end
end;


## Test 1 — Periodic box: discrete consistency

Deterministic multi-mode + Gaussian-blob density in a triply periodic $64^3$ box
(8 meshblocks). The solver must satisfy the 7-point discrete Poisson equation
$L[\Phi] = 4\pi G(\rho - \bar\rho)$ **to machine precision** — the FFT inverse is exact,
so any residual is a bug in the gather/scatter, kernel, or ghost fill.

In [ ]:
diag, _ = run_athena("fft_poisson.athinput")
println.(diag);
err = parse_diag(diag[1])
@printf("solver-reported residual (double precision): max = %.2e   ✓ machine precision\n",
        err["max_rel_all"])


In [ ]:
# Independent check: recompute the Laplacian residual in Julia from the .bin files.
# (float32 snapshot data => expect ~1e-6..1e-5 here, NOT 1e-14; an actual solver bug
#  would show up as O(1).)
w = read_athenak_bin(joinpath(RUNDIR, "bin", "FFTGravityTest.hydro_w.00000.bin"))
p = read_athenak_bin(joinpath(RUNDIR, "bin", "FFTGravityTest.grav_phi.00000.bin"))
rho, phi = w.vars["dens"], p.vars["grav_phi"]
dx = (w.mesh["x1max"] - w.mesh["x1min"]) / w.Nx[1]         # cubic cells here
lap = (circshift(phi,(-1,0,0)) .+ circshift(phi,(1,0,0)) .+
       circshift(phi,(0,-1,0)) .+ circshift(phi,(0,1,0)) .+
       circshift(phi,(0,0,-1)) .+ circshift(phi,(0,0,1)) .- 6phi) ./ dx^2
rhs = (rho .- sum(rho)/length(rho))                        # four_pi_G = 1
@printf("independent Julia residual (float32 data): max = %.2e\n",
        maximum(abs.(lap .- rhs)) / maximum(abs.(rhs)))


In [ ]:
kmid = w.Nx[3] ÷ 2
fig = Figure(size=(900, 380))
for (col, f, ttl) in ((1, rho[:, :, kmid] .- 1.0, "ρ − ρ₀   (z = 0)"),
                      (2, phi[:, :, kmid] .- sum(phi)/length(phi), "Φ − ⟨Φ⟩   (z = 0)"))
    lim = maximum(abs, f)
    ax = Axis(fig[1, 2col-1]; xlabel="x", ylabel="y", title=ttl, aspect=DataAspect())
    hm = heatmap!(ax, w.x, w.y, f; colormap=:RdBu, colorrange=(-lim, lim))
    Colorbar(fig[1, 2col], hm)
end
fig


## Test 2 — Shear bookkeeping: exact at shear-periodic instants

In a $q=1.5$, $\Omega=1$ box, the shear-periodic time is $T_{\rm sh}=L_y/(q\Omega L_x)=2/3$.
At $t = n\,T_{\rm sh}$ the accumulated shear phase `qomt` vanishes and the whole shear code
path (roll → shifted kernel → unroll → shear ghosts) must reduce to the plain periodic
solve **exactly**. At a generic phase the residual is finite (remap interpolation) — that's
expected; see the metric caveat at the end.

In [ ]:
d_gen, _ = run_athena("fft_poisson_shear.athinput")                       # t0 = 0.37, qomt = 0.555
d_per, _ = run_athena("fft_poisson_shear.athinput";
                      overrides=["problem/time0=0.6666666666666667"])     # t0 = Tsh, qomt = 0
@printf("generic phase (qomt=0.555): max_rel = %.3e   (finite: remap error)\n",
        parse_diag(d_gen[1])["max_rel_all"])
@printf("periodic instant (qomt=0) : max_rel = %.3e   ✓ machine precision\n",
        parse_diag(d_per[1])["max_rel_all"])


## Test 3 — Shearing-wave analytic solution: order of accuracy

`profile = shwave` initializes a **single Fourier mode of the rolled frame**
$(n_1,n_2,n_3)$, which in the current frame is a slanted wave:
$\rho = \rho_0[1 + A\cos(2\pi(n_1 x/L_x + n_2(y+\theta x)/L_y + n_3 z/L_z))]$
with $\theta = q\Omega t_s$. Its exact discrete solution is the same wave with amplitude
$$\Phi_{\rm amp} = \frac{4\pi G\,\rho_0 A}{D},\qquad
D = \tfrac{2\cos k_x'\Delta x - 2}{\Delta x^2} + \tfrac{2\cos k_y\Delta y - 2}{\Delta y^2}
  + \tfrac{2\cos k_z\Delta z - 2}{\Delta z^2},\quad
k_x'\Delta x = (n_1 + \theta\tfrac{L_x}{L_y}n_2)\tfrac{2\pi}{N_x}.$$
The pointwise $\Phi$ error measures the full pipeline and converges at the **remap
interpolation order** (3rd for `ppmx`, 2nd for `plm`/`dc`).

In [ ]:
# Independent Julia-side comparison at 64³ (ppmx): build the analytic field ourselves
run_athena("fft_poisson_shear.athinput";
           overrides=["problem/profile=shwave", "job/basename=FFTGravityShwave"])
p3 = read_athenak_bin(joinpath(RUNDIR, "bin", "FFTGravityShwave.grav_phi.00000.bin"))
q, Om, t0, amp, rho0, fpg = 1.5, 1.0, 0.37, 0.1, 1.0, 1.0
n1w, n2w, n3w = 1, 1, 1
Lx = p3.mesh["x1max"]-p3.mesh["x1min"]; Ly = p3.mesh["x2max"]-p3.mesh["x2min"]
Lz = p3.mesh["x3max"]-p3.mesh["x3min"]
Tsh  = Ly/(q*Om*Lx)
qomt = q*Om*(t0 - floor(t0/Tsh)*Tsh)
dx1, dx2, dx3 = Lx/p3.Nx[1], Ly/p3.Nx[2], Lz/p3.Nx[3]
D = (2cos((n1w + qomt*(Lx/Ly)*n2w)*2π/p3.Nx[1]) - 2)/dx1^2 +
    (2cos(n2w*2π/p3.Nx[2]) - 2)/dx2^2 + (2cos(n3w*2π/p3.Nx[3]) - 2)/dx3^2
Φamp = amp*rho0*fpg/D          # NEGATIVE: wells align with density crests
ana  = [Φamp*cos(2π*(n1w*x/Lx + n2w*(y + qomt*x)/Ly + n3w*z/Lz))
        for x in p3.x, y in p3.y, z in p3.z]
ϕ = p3.vars["grav_phi"]
errmax = maximum(abs.((ϕ .- sum(ϕ)/length(ϕ)) .- (ana .- sum(ana)/length(ana)))) / abs(Φamp)
@printf("independent Julia phi error (64³ ppmx, float32 data): %.4e  (pgen: 1.264e-5)\n", errmax)


In [ ]:
# Convergence scan: parse the pgen's double-precision numbers for 3 remaps × 3 resolutions
res = Dict{String,Vector{Tuple{Int,Float64}}}()
for rm in ("ppmx", "plm", "dc"), N in (32, 64, 128)
    ov = ["problem/profile=shwave", "gravity/fft_remap=$rm", "job/basename=ShwConv",
          "mesh/nx1=$N", "mesh/nx2=$N", "mesh/nx3=$N",
          "meshblock/nx1=$N", "meshblock/nx2=$N", "meshblock/nx3=$N"]
    d, _ = run_athena("fft_poisson_shear.athinput"; overrides=ov)
    e = parse_diag(d[end])["max_rel"]
    push!(get!(res, rm, Tuple{Int,Float64}[]), (N, e))
    @printf("%-5s N=%3d  max_rel = %.3e\n", rm, N, e)
end
for rm in ("ppmx", "plm", "dc")
    r = res[rm]
    orders = [log2(r[i][2]/r[i+1][2]) for i in 1:length(r)-1]
    @printf("%-5s measured orders per doubling: %s\n", rm,
            join([@sprintf("%.2f", o) for o in orders], ", "))
end


In [ ]:
fig = Figure(size=(620, 460))
ax = Axis(fig[1, 1]; xscale=log10, yscale=log10,
          xlabel="N  (cells per side)", ylabel="max |Φ − Φ_analytic| / |Φ_amp|",
          title="Shearing-wave pointwise Φ error",
          xticks=([32, 64, 128], ["32", "64", "128"]))
for (rm, c, lbl) in (("ppmx", C1, "ppmx (3rd order)"),
                     ("plm", C2, "plm (2nd order)"), ("dc", C3, "dc (2nd order)"))
    r = res[rm]
    scatterlines!(ax, Float64.(first.(r)), last.(r); color=c, label=lbl, markersize=10)
end
Ns = [32.0, 128.0]
lines!(ax, Ns, 1.4e-2 .* (Ns ./ 32) .^ -2; linestyle=:dash, color=:gray, label="N⁻²")
lines!(ax, Ns, 3.5e-4 .* (Ns ./ 32) .^ -3; linestyle=:dash, color=:gray, label="N⁻³")
axislegend(ax; position=:lb, framevisible=false)
fig


## Test 4 — Open vertical boundary: slab potential

`vert_bc = open` uses the Koyama & Ostriker (2009) two-solve method (periodic +
antiperiodic vertical harmonics with screening weights $\frac12(1 \mp e^{-k_\perp L_z})$),
which cancels all vertical periodic images analytically. For a horizontally uniform
$\rho = \rho_0\,\mathrm{sech}^2(z/H)$ slab only the $k_\perp = 0$ column is active, where the
weights are exact ⇒ the **interior** residual is machine precision, and $\Phi(z)$ should match
the analytic infinite-slab potential $4\pi G \rho_0 H^2 \ln\cosh(z/H)$ up to a constant.

In [ ]:
d_open, _ = run_athena("fft_poisson_open.athinput")
println.(d_open);
po = read_athenak_bin(joinpath(RUNDIR, "bin", "FFTGravityOpen.grav_phi.00000.bin"))
H, rho0, fpg = 0.05, 1.0, 1.0
zc = 0.5*(po.mesh["x3min"] + po.mesh["x3max"])
ϕz  = po.vars["grav_phi"][1, 1, :]
ana = fpg*rho0*H^2 .* log.(cosh.((po.z .- zc) ./ H))
ϕc, anac = ϕz .- sum(ϕz)/length(ϕz), ana .- sum(ana)/length(ana)
@printf("independent Julia slab error: max_rel = %.3e\n",
        maximum(abs.(ϕc .- anac)) / maximum(abs.(anac)))
fig = Figure(size=(640, 560))
ax1 = Axis(fig[1, 1]; ylabel="Φ(z) − ⟨Φ⟩", title="sech² slab, H = Lz/20, open vertical BC")
lines!(ax1, po.z, anac; color=C2, linewidth=2.5, label="analytic (infinite slab)")
scatter!(ax1, po.z[1:2:end], ϕc[1:2:end]; color=C1, markersize=7, label="FFT solver")
axislegend(ax1; framevisible=false)
ax2 = Axis(fig[2, 1]; yscale=log10, xlabel="z", ylabel="rel. error", height=140,
           yticks=([1e-5, 1e-4, 1e-3], ["10⁻⁵", "10⁻⁴", "10⁻³"]))
lines!(ax2, po.z, max.(abs.(ϕc .- anac) ./ maximum(abs, anac), 1e-16); color=C1)
fig


## Test 5 — Swing amplification (gravity feeds back on the dynamics)

The first test where self-gravity acts *dynamically*. A leading shearing wave
($k_{x0}<0$, $k_y>0$, $k_z=0$) is wound up by the shear, $k_x(t) = k_{x0} + q\Omega k_y t$;
as it swings through $k_x = 0$ self-gravity transiently amplifies it
(Goldreich & Lynden-Bell 1965; Julian & Toomre 1966; Toomre 1981).

Linearizing the isothermal shearing-box equations for a single shearing wave
($\delta = \delta\rho/\rho_0$, $w_{x,y} = i\hat v_{x,y}$, so $v = w\sin(\mathbf{k}\cdot\mathbf{x})$
while $\delta$ stays $\propto\cos$):

$$\delta' = -(k_x w_x + k_y w_y),\quad
w_x' = 2\Omega w_y + k_x S,\quad
w_y' = -(2-q)\Omega w_x + k_y S,\quad
S = \delta\left(c_s^2 + \frac{4\pi G\rho_0}{D}\right)$$

with $D$ the discrete Laplacian eigenvalue (the solver's own kernel; $\to -k^2$ in the
continuum). Setting $4\pi G = 0$ recovers pure hydrodynamic winding — the control.

The `swing` pgen initializes $\rho = \rho_0[1 + A\cos(k_{x0}x + k_y y)]$, $v' = 0$, and its
history output projects the solution onto the *instantaneous* $\mathbf{k}(t)$, yielding
$\delta$, $w_x$, $w_y$ directly. Parameters sit in the epicyclic-resonant regime
($k_y = 1 = \kappa$, $4\pi G\rho_0 = 1.9$ just below the marginal $k_y^2c_s^2 + \kappa^2 = 2$),
where swing is strong but every box mode stays Jeans-stable.

In [ ]:
# --- linear swing ODE (RK4; no external ODE package needed) -------------------------
function swing_rhs(u, t, p)
    δ, wx, wy = u
    kx = p.kx0 + p.q*p.Ω*p.ky*t
    D  = p.discrete ? (2cos(kx*p.dx)-2)/p.dx^2 + (2cos(p.ky*p.dy)-2)/p.dy^2 :
                      -(kx^2 + p.ky^2)
    S  = δ*(p.cs^2 + p.fpg*p.ρ0/D)
    return (-(kx*wx + p.ky*wy), 2*p.Ω*wy + kx*S, -(2 - p.q)*p.Ω*wx + p.ky*S)
end

function swing_integrate(p; t0=0.0, t1=8.0, nsteps=400_000)
    h = (t1-t0)/nsteps; u = (p.amp, 0.0, 0.0); t = t0
    ts=[t]; δs=[u[1]]; wxs=[u[2]]; wys=[u[3]]
    for _ in 1:nsteps
        k1 = swing_rhs(u, t, p);              k2 = swing_rhs(u .+ (h/2).*k1, t+h/2, p)
        k3 = swing_rhs(u .+ (h/2).*k2, t+h/2, p); k4 = swing_rhs(u .+ h.*k3, t+h, p)
        u = u .+ (h/6).*(k1 .+ 2 .*k2 .+ 2 .*k3 .+ k4); t += h
        push!(ts,t); push!(δs,u[1]); push!(wxs,u[2]); push!(wys,u[3])
    end
    ts, δs, wxs, wys
end

"Read an AthenaK .hst history file into a matrix (rows = times, cols = variables)."
function read_hst(f)
    rows = Float64[]; ncol = 0
    for ln in eachline(f)
        startswith(strip(ln), "#") && continue
        v = parse.(Float64, split(ln)); ncol = length(v); append!(rows, v)
    end
    permutedims(reshape(rows, ncol, :))
end
lininterp(xs, ys, x) = (i = searchsortedlast(xs, x);
    i < 1 ? ys[1] : i >= length(xs) ? ys[end] :
    ys[i] + (ys[i+1]-ys[i])*(x-xs[i])/(xs[i+1]-xs[i]));


In [ ]:
# Sanity-check the reference ODE itself: with ky → 0 and no rotation it must reduce to
# the Jeans oscillation δ'' = -(k²cs² - 4πGρ0)δ.
println("ODE vs Jeans dispersion relation (ky → 0):")
for (k, fpg) in ((2.0, 0.0), (2.0, 2.0), (1.0, 3.0))
    pj = (kx0=k, ky=1e-9, q=0.0, Ω=0.0, cs=1.0, ρ0=1.0, fpg=fpg,
          dx=1e-6, dy=1e-6, discrete=false, amp=1.0)
    _, δj, _, _ = swing_integrate(pj; t1=6.0)
    ω² = k^2 - fpg
    exact = ω² > 0 ? cos(sqrt(ω²)*6) : cosh(sqrt(-ω²)*6)
    @printf("  k=%.1f 4πG=%.1f  ω²=%+.2f   ODE %+.6f   analytic %+.6f\n",
            k, fpg, ω², δj[end], exact)
end


In [ ]:
# run the simulation (≈35 s at 128²; delete the .hst to force a re-run)
if !isfile(joinpath(RUNDIR, "SwingSG.user.hst"))
    run_athena("swing_selfgrav.athinput")
end
h = read_hst(joinpath(RUNDIR, "SwingSG.user.hst"))
t, dcos, dsin, vxs, vys, kxky = h[:,1], h[:,3], h[:,4], h[:,5], h[:,6], h[:,7]

p = (kx0=-6.0, ky=1.0, q=1.5, Ω=1.0, cs=1.0, ρ0=1.0, fpg=1.9,
     dx=2π/128, dy=2π/128, discrete=true, amp=1.0e-4)
ts, δs, wxo_, wyo_ = swing_integrate(p)
δo  = [lininterp(ts, δs,   tt) for tt in t]
wxo = [lininterp(ts, wxo_, tt) for tt in t]
wyo = [lininterp(ts, wyo_, tt) for tt in t]

@printf("t_swing (kx=0)       : %.3f   (history crosses at %.3f)\n",
        -p.kx0/(p.q*p.Ω*p.ky), t[argmin(abs.(kxky))])
@printf("peak |δ|/δ0  sim/ODE : %.3f / %.3f\n",
        maximum(abs,dcos)/p.amp, maximum(abs,δo)/p.amp)
@printf("L2 error vs linear theory: δ %.2e   wx %.2e   wy %.2e\n",
        sqrt(sum((dcos.-δo).^2)/length(t))/maximum(abs,δo),
        sqrt(sum((vxs.-wxo).^2)/length(t))/maximum(abs,wxo),
        sqrt(sum((vys.-wyo).^2)/length(t))/maximum(abs,wyo))
@printf("max |d_sin| / peak|δ|    : %.2e   (phase locking; linear theory says 0)\n",
        maximum(abs,dsin)/maximum(abs,δo))


In [ ]:
fig = Figure(size=(760, 620))
ax1 = Axis(fig[1, 1]; ylabel="δ / δ₀", title="Swing amplification of a leading shearing wave")
vlines!(ax1, [-p.kx0/(p.q*p.Ω*p.ky)]; color=(:gray, 0.6), linestyle=:dash)
lines!(ax1, t, δo ./ p.amp; color=C2, linewidth=3, label="linear theory (ODE)")
scatter!(ax1, t[1:8:end], dcos[1:8:end] ./ p.amp; color=C1, markersize=6,
         label="AthenaK + FFT self-gravity")
text!(ax1, -p.kx0/(p.q*p.Ω*p.ky), -3.2; text="  kₓ = 0 (swing)", color=:gray, fontsize=11)
axislegend(ax1; position=:lt, framevisible=false)

ax2 = Axis(fig[2, 1]; ylabel="wₓ , w_y", xlabel="t   (Ω⁻¹)")
lines!(ax2, t, wxo; color=C2, linewidth=2.5, label="wₓ theory")
scatter!(ax2, t[1:12:end], vxs[1:12:end]; color=C1, markersize=6, label="wₓ sim")
lines!(ax2, t, wyo; color=C3, linewidth=2.5, linestyle=:dash, label="w_y theory")
scatter!(ax2, t[1:12:end], vys[1:12:end]; color=C1, marker=:rect, markersize=6,
         label="w_y sim")
axislegend(ax2; position=:lt, framevisible=false, nbanks=2)
rowsize!(fig.layout, 2, Relative(0.38))
fig


In [ ]:
# Is the residual error from gravity or from the hydro scheme? Repeat the resolution
# study with gravity OFF and compare each run to its own linear theory. Runs must exist:
#   for N in 64 128 256: athena -i swing_selfgrav.athinput job/basename=SwingN$N ...
#   and the same with hydro_srcterms/self_gravity=false -> SwingNoG$N
for (tag, fpg, pre) in (("gravity ON ", 1.9, "SwingN"), ("gravity OFF", 0.0, "SwingNoG"))
    println("--- ", tag)
    prev = NaN
    for N in (64, 128, 256)
        f = joinpath(RUNDIR, "$pre$N.user.hst")
        isfile(f) || (println("  (missing $(basename(f)) — run it first)"); continue)
        hh = read_hst(f); tt, dd = hh[:,1], hh[:,3]
        pp = (kx0=-6.0, ky=1.0, q=1.5, Ω=1.0, cs=1.0, ρ0=1.0, fpg=fpg,
              dx=2π/N, dy=2π/N, discrete=true, amp=1.0e-4)
        tsx, δsx, _, _ = swing_integrate(pp)
        δox = [lininterp(tsx, δsx, x) for x in tt]
        l2 = sqrt(sum((dd .- δox).^2)/length(tt))/maximum(abs, δox)
        @printf("  N=%-4d peak|δ|/δ0 = %-7.4f  L2 = %-9.3e %s\n", N,
                maximum(abs,dd)/pp.amp, l2,
                isnan(prev) ? "" : @sprintf("order %.2f", log2(prev/l2)))
        prev = l2
    end
end
println("\nAt 256² the two L2 errors agree to ~3%: an error the same size with and without")
println("the Poisson solve cannot come from the Poisson solve — it is the hydro scheme.")


## Playground — your own initial conditions

Everything in the `<problem>` block (and beyond) is overridable per run. Useful knobs:

| override | effect |
|---|---|
| `problem/profile=modes\|shwave\|slab` | density field type |
| `problem/amp=0.3`, `problem/blob_amp=0` | mode amplitude / remove the blob |
| `problem/n1=3 problem/n2=2 problem/n3=1` | shwave mode numbers (rolled frame) |
| `problem/time0=0.5` | shear phase at the solve (`qomt = qΩ·(t mod Tsh)`) |
| `problem/rho0=…`, `problem/slab_h=…` | background density, slab scale height |
| `gravity/fft_remap=dc\|plm\|ppmx` | y-remap interpolation order |
| `gravity/vert_bc=periodic\|open` | vertical boundary (needs matching x3 BCs) |
| `gravity/four_pi_G=…` | gravitational constant |
| `mesh/nx1=… meshblock/nx1=…` (etc.) | resolution / block decomposition |

For the swing test (`swing_selfgrav.athinput`):

| override | effect |
|---|---|
| `problem/nwx=-8 problem/nwy=1` | initial wavevector; $t_{\rm swing} = -n_{wx}/(q\,n_{wy})$ |
| `gravity/four_pi_G=1.9` | gravity strength (marginal at $k_y^2c_s^2+\kappa^2 = 2$) |
| `shearing_box/qshear=1.0` | shear rate (flat rotation curve; changes $\kappa^2 = 2(2-q)\Omega^2$) |
| `hydro_srcterms/self_gravity=false` | control run: no amplification |
| `problem/amp=1e-5` | smaller amplitude (stays linear longer) |

Change any of these and re-integrate the ODE with matching `p` to compare.

Snapshots land in `validation/run/bin/<basename>.{hydro_w,grav_phi}.*.bin` and history in
`<basename>.user.hst`; give each experiment its own `job/basename=…`.

Below: an example sweep — how the residual varies with shear phase across one period.

In [ ]:
t0s = range(0.0, 2/3; length=9)
errs = Float64[]
for t0 in t0s
    d, _ = run_athena("fft_poisson_shear.athinput";
                      overrides=["problem/time0=$t0", "job/basename=PhaseSweep"])
    push!(errs, parse_diag(d[1])["max_rel_all"])
end
fig = Figure(size=(620, 420))
ax = Axis(fig[1, 1]; yscale=log10, xlabel="t₀ / T_sh   (shear phase)",
          ylabel="max Laplacian residual",
          title="Residual vs shear phase (64³, modes profile, ppmx)",
          yticks=([1e-14, 1e-11, 1e-8, 1e-5, 1e-2],
                  ["10⁻¹⁴", "10⁻¹¹", "10⁻⁸", "10⁻⁵", "10⁻²"]))
scatterlines!(ax, collect(t0s) ./ (2/3), max.(errs, 1e-16); color=C1, markersize=10)
fig


## Reading the numbers: which metric means what

- **Laplacian residual** (`# FFT-POISSON ERRORS`): machine precision *only* when the remap
  is trivial (no shear, or `qomt = 0`) and BCs are exact. Under shear it is dominated by
  (remap interpolation error)/Δ², improving only ≈1st order — *by construction, not a bug*
  (a spectral-interpolation remap would zero it). Sharp features (the blob) can stall the
  max-norm via the PPMX limiter.
- **Pointwise Φ error vs analytic** (shwave, slab): the correct order-of-accuracy metric —
  converges at the remap order (Test 3) or the BC/discretization limit (Test 4).
- **`*_int` norms** exclude layers whose ghost values are themselves approximate
  (shear-remapped x1 layers; extrapolated x3 layers in open mode).
- **Swing test (Test 5)** is limited by *hydrodynamic* truncation error, not by the Poisson
  solve — it converges with resolution, and the phase-locking diagnostic (`d_sin` ~ 1e-13)
  isolates the shear bookkeeping, which stays exact regardless of resolution.

**Remaining work** on the solver: multi-rank support (`MPI_Allgatherv` at the documented
seam in the gather/scatter), a Python regression harness mirroring
`tst/test_suite/multigrid/`, and then the FFT-root + multigrid-refinement hybrid that is
the actual goal — self-gravity with mesh refinement in a shearing box.